### Tạo dữ liệu và chia batch

In [15]:
import numpy as np

X = np.arange(10, dtype=float).reshape(-1, 1) / 10
y = 3 * X[:, 0] + 2

batch_size = 4
rng = np.random.default_rng(42)
indices = rng.permutation(len(y))

for start in range(0, len(y), batch_size):
    batch_idx = indices[start:start + batch_size]

    X_batch = X[batch_idx]
    y_batch = y[batch_idx]

    print("Chỉ số:", batch_idx)
    print("Shape:", X_batch.shape, y_batch.shape)

Chỉ số: [5 6 0 7]
Shape: (4, 1) (4,)
Chỉ số: [3 2 4 9]
Shape: (4, 1) (4,)
Chỉ số: [1 8]
Shape: (2, 1) (2,)


### MSE và gradient

In [16]:
def mean_squared_error(y_true, y_pred):
    return np.mean((y_pred - y_true) ** 2)


def linear_gradients(X, y_true, y_pred):
    error = y_pred - y_true
    dw = 2 * X.T @ error / len(y_true)
    db = 2 * error.mean()
    return dw, db

### train bằng mini-batch

In [17]:
class MiniBatchLinearRegression:
    def __init__(
        self,
        learning_rate=0.1,
        n_epochs=1000,
        batch_size=4,
        random_state=42,
    ):
        self.learning_rate = learning_rate
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.random_state = random_state

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        self.b = 0.0
        self.loss_history = []
        self.n_updates = 0

        rng = np.random.default_rng(self.random_state)

        for _ in range(self.n_epochs):
            indices = rng.permutation(len(y))

            for start in range(0, len(y), self.batch_size):
                batch_idx = indices[start:start + self.batch_size]
                X_batch = X[batch_idx]
                y_batch = y[batch_idx]

                y_pred = self.predict(X_batch)
                dw, db = linear_gradients(X_batch, y_batch, y_pred)

                self.w -= self.learning_rate * dw
                self.b -= self.learning_rate * db
                self.n_updates += 1

            epoch_loss = mean_squared_error(y, self.predict(X))
            self.loss_history.append(epoch_loss)

        return self

    def predict(self, X):
        return X @ self.w + self.b

### Train và kiểm tra

In [18]:
model = MiniBatchLinearRegression(
    learning_rate=0.1,
    n_epochs=1000,
    batch_size=4,
)

model.fit(X, y)

print("w:", model.w.round(4))
print("b:", round(model.b, 4))
print("MSE cuối:", model.loss_history[-1])
print("Số lần cập nhật:", model.n_updates)

X_new = np.array([[0.25], [0.75]])
print("Dự đoán:", model.predict(X_new).round(4))

w: [3.]
b: 2.0
MSE cuối: 6.113672015462842e-31
Số lần cập nhật: 3000
Dự đoán: [2.75 4.25]


## train trên dữ liệu có nhiễu và đánh giá bằng MAE, MSE, RMSE

### Tạo 500 mẫu và chia train/test

In [19]:
rng = np.random.default_rng(42)

X = rng.uniform(0, 1, size=(500, 1))
noise = rng.normal(loc=0, scale=0.3, size=500)
y = 3 * X[:, 0] + 2 + noise

indices = rng.permutation(len(y))
train_idx = indices[:400]
test_idx = indices[400:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (400, 1) (400,)
Test: (100, 1) (100,)


### ba metric

In [20]:
def mean_absolute_error(y_true, y_pred):
    return np.mean(np.abs(y_pred - y_true))


def root_mean_squared_error(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

### Train rồi đánh giá

In [21]:
model = MiniBatchLinearRegression(
    learning_rate=0.1,
    n_epochs=500,
    batch_size=32,
    random_state=42,
)

model.fit(X_train, y_train)

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

print("w học được:", model.w.round(3))
print("b học được:", round(model.b, 3))

print("MAE train:", mean_absolute_error(y_train, train_pred))
print("MAE test:", mean_absolute_error(y_test, test_pred))
print("MSE test:", mean_squared_error(y_test, test_pred))
print("RMSE test:", root_mean_squared_error(y_test, test_pred))

w học được: [2.985]
b học được: 1.994
MAE train: 0.24523331332502474
MAE test: 0.22120076803178676
MSE test: 0.08030667597776522
RMSE test: 0.2833843255682382


### Thêm một lỗi lớn để hiểu MAE và RMSE

In [22]:
y_example = np.array([10.0, 10.0, 10.0, 10.0])

pred_a = np.array([11.0, 11.0, 11.0, 11.0])
pred_b = np.array([11.0, 11.0, 11.0, 20.0])

for name, predictions in [("A", pred_a), ("B", pred_b)]:
    print(
        name,
        "MAE =", round(mean_absolute_error(y_example, predictions), 3),
        "MSE =", round(mean_squared_error(y_example, predictions), 3),
        "RMSE =", round(root_mean_squared_error(y_example, predictions), 3),
    )

A MAE = 1.0 MSE = 1.0 RMSE = 1.0
B MAE = 3.25 MSE = 25.75 RMSE = 5.074


## Linear Regression nhiều đặc trưng và chuẩn hóa dữ liệu bằng NumPy

### Tạo dữ liệu có hai đặc trưng

In [23]:
rng = np.random.default_rng(42)

x1 = rng.uniform(0, 1, size=500)
x2 = rng.uniform(0, 1000, size=500)

X = np.column_stack([x1, x2])
noise = rng.normal(0, 0.3, size=500)

y = 3 * x1 + 0.005 * x2 + 2 + noise

indices = rng.permutation(len(y))
train_idx = indices[:400]
test_idx = indices[400:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

print("Shape:", X_train.shape)  # (400, 2)
print("Ba mẫu đầu:\n", X_train[:3])

Shape: (400, 2)
Ba mẫu đầu:
 [[8.53403073e-01 4.82572509e+02]
 [5.20672402e-01 5.14629342e+02]
 [9.04407270e-02 9.01641854e+02]]


### chuẩn hóa theo từng cột

In [24]:
class StandardScaler:
    def fit(self, X):
        self.mean = X.mean(axis=0)
        self.scale = X.std(axis=0)

        # Cột có mọi giá trị giống nhau sẽ có std = 0
        self.scale = np.where(self.scale == 0, 1.0, self.scale)
        return self

    def transform(self, X):
        return (X - self.mean) / self.scale

### Học cách chuẩn hóa từ train rồi áp dụng

In [25]:
scaler = StandardScaler().fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Mean gốc:", scaler.mean.round(3))
print("Std gốc:", scaler.scale.round(3))

print("Mean train sau chuẩn hóa:",
      X_train_scaled.mean(axis=0).round(6))
print("Std train sau chuẩn hóa:",
      X_train_scaled.std(axis=0).round(6))

Mean gốc: [4.91000e-01 4.97382e+02]
Std gốc: [2.83000e-01 2.97595e+02]
Mean train sau chuẩn hóa: [0. 0.]
Std train sau chuẩn hóa: [1. 1.]


### Train trên dữ liệu đã chuẩn hóa

In [26]:
model = MiniBatchLinearRegression(
    learning_rate=0.1,
    n_epochs=500,
    batch_size=32,
    random_state=42,
)

model.fit(X_train_scaled, y_train)

test_pred = model.predict(X_test_scaled)

print("w trên dữ liệu chuẩn hóa:", model.w.round(3))
print("b trên dữ liệu chuẩn hóa:", round(model.b, 3))
print("MAE test:", mean_absolute_error(y_test, test_pred))
print("RMSE test:", root_mean_squared_error(y_test, test_pred))

w trên dữ liệu chuẩn hóa: [0.862 1.468]
b trên dữ liệu chuẩn hóa: 5.962
MAE test: 0.2292119500298964
RMSE test: 0.3028550426009105


### Đổi trọng số về đơn vị gốc

In [27]:
w_original = model.w / scaler.scale
b_original = model.b - scaler.mean @ w_original

print("w theo đơn vị gốc:", w_original.round(4))
print("b theo đơn vị gốc:", round(b_original, 4))

# Hai cách tính phải cho cùng dự đoán
pred_original = X_test @ w_original + b_original
print("Hai cách khớp nhau:",
      np.allclose(test_pred, pred_original))

X_new = np.array([[0.5, 800.0]])

prediction_1 = model.predict(scaler.transform(X_new))
prediction_2 = X_new @ w_original + b_original

print(prediction_1, prediction_2)

w theo đơn vị gốc: [3.0421 0.0049]
b theo đơn vị gốc: 2.0139
Hai cách khớp nhau: True
[7.48248695] [7.48248695]
